In [1]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_mensajeria = config['MENSAJERIA_OLTP']
    config_etl = config['ETL_PROCESS']

# Construct the database URL
url_mensajeria = (f"{config_mensajeria['drivername']}://{config_mensajeria['user']}:{config_mensajeria['password']}@{config_mensajeria['host']}:"
          f"{config_mensajeria['port']}/{config_mensajeria['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_mensajeria)
etl_conn = create_engine(url_etl)

In [ ]:
fechas = pd.read_sql_table('mensajeria_estadosservicio',mensajeria)
estados_servicios = pd.read_sql_table('mensajeria_estado',mensajeria)
servicio = pd.read_sql_table('mensajeria_servicio',mensajeria)

fechas = fechas.drop(columns=['foto','observaciones','es_prueba','foto_binary'])

fechas['day_of_week'] = fechas['fecha'].dt.weekday
fechas['year'] = fechas['fecha'].dt.year
fechas['month'] = fechas['fecha'].dt.month

# fechas['Estado'] = mensajeria_estado['nombre']
estados_servicios.loc[0,"nombre"] = "Recogido en origen"
estados_servicios.loc[3,"nombre"] = "Cerrado"

fechas = fechas.merge(
    estados_servicios[['id','nombre']],
    left_on ='estado_id',
    right_on='id',
    how='left'
)
fechas.rename(columns={'nombre':'estado'},inplace=True)

fechas.drop(columns=['id_y'],inplace=True)

servicio.drop(columns=['descripcion', 'nombre_solicitante', 'fecha_solicitud',
       'hora_solicitud', 'fecha_deseada', 'hora_deseada', 'nombre_recibe',
       'telefono_recibe', 'descripcion_pago', 'ida_y_regreso', 'activo',
       'novedades', 'cliente_id', 'destino_id', 'mensajero_id', 'origen_id',
       'tipo_pago_id', 'tipo_servicio_id', 'tipo_vehiculo_id', 'usuario_id',
       'prioridad', 'ciudad_destino_id', 'ciudad_origen_id',
       'hora_visto_por_mensajero', 'visto_por_mensajero',
       'descripcion_multiples_origenes', 'mensajero2_id', 'mensajero3_id',
       'multiples_origenes', 'asignar_mensajero', 'es_prueba',
       'descripcion_cancelado'],inplace=True)

fechas

servicio = servicio.merge(
    fechas,
    left_on='id',
    right_on='servicio_id',
    how='left'
)

servicio.drop(columns=['estado_id'],inplace=True)

servicio["fecha_hora"] = pd.to_datetime(
    servicio["fecha"].dt.date.astype(str) +
    " " +
    servicio["hora"].astype(str),
    errors="coerce"
)

servicio

In [4]:
dim_fechahora = (
    servicio[["fecha_hora"]]
    .drop_duplicates()
    .sort_values("fecha_hora")
    .reset_index(drop=True)
)

dim_fechahora["fecha_hora_key"] = (
    dim_fechahora.index + 1
)

dim_fechahora["año"] = dim_fechahora["fecha_hora"].dt.year
dim_fechahora["mes"] = dim_fechahora["fecha_hora"].dt.month
dim_fechahora["dia"] = dim_fechahora["fecha_hora"].dt.day
dim_fechahora["hora"] = dim_fechahora["fecha_hora"].dt.hour
dim_fechahora["minuto"] = dim_fechahora["fecha_hora"].dt.minute
dim_fechahora["dia_de_la_semana"] = dim_fechahora["fecha_hora"].dt.day_name()

dim_fechahora

dim_fechahora.to_sql("dim_fechahora", etl_conn, if_exists="replace", index_label="key_dim_fechahora")

11

In [5]:
trans_servicio = servicio.merge(
    dim_fechahora[['fecha_hora_key','fecha_hora']],
    on='fecha_hora',
    how='left'
)

trans_servicio.drop(columns=['id_x','fecha','hora','servicio_id','day_of_week','year','month'],inplace=True)
trans_servicio


,id,estado,fecha_hora,fecha_hora_key
0,34,Iniciado,2023-10-26 09:46:03,91
1,35,Iniciado,2023-10-26 11:18:14,92
2,35,Con mensajero Asignado,2023-10-28 14:43:08,93
3,35,Recogido en origen,2023-10-28 19:45:18,95
4,35,Entregado en destino,2023-12-07 01:25:34,164
...,...,...,...,...
128397,28437,Con mensajero Asignado,2024-08-31 10:44:39,126818
128398,28437,Iniciado,2024-08-31 10:40:33,126812
128399,28437,Recogido en origen,2024-08-31 11:07:02,126855
128400,28437,Entregado en destino,2024-08-31 11:27:20,126878


In [6]:
tiempos = trans_servicio.pivot_table(
    index='id',
    columns='estado',
    values='fecha_hora',
    aggfunc='first'
).reset_index()

tiempos['minutos_asignacion'] = (
    tiempos['Con mensajero Asignado']
    - tiempos['Iniciado']
).dt.total_seconds() / 60

tiempos['minutos_recogida'] = (
    tiempos['Recogido en origen']
    - tiempos['Con mensajero Asignado']
).dt.total_seconds() / 60

tiempos['minutos_entrega'] = (
    tiempos['Entregado en destino']
    - tiempos['Recogido en origen']
).dt.total_seconds() / 60

tiempos['minutos_cierre'] = (
    tiempos['Cerrado']
    - tiempos['Entregado en destino']
).dt.total_seconds() / 60

tiempos



estado,id,Cerrado,Con mensajero Asignado,Con novedad,Entregado en destino,Iniciado,Recogido en origen,minutos_asignacion,minutos_recogida,minutos_entrega,minutos_cierre
0,7,2023-10-31 12:16:00,2023-10-13 17:51:20,NaT,2023-10-31 17:07:55,2023-09-19 16:22:18,NaT,34649.033333,NaN,NaN,-291.916667
1,8,NaT,2023-12-20 20:14:43,NaT,2024-04-09 16:08:35,2023-09-19 16:30:05,2024-02-14 15:34:18,132704.633333,80359.583333,79234.283333,NaN
2,9,NaT,2023-12-28 19:33:01,NaT,NaT,2023-09-19 16:30:05,NaT,144182.933333,NaN,NaN,NaN
3,10,NaT,2023-12-28 19:33:07,NaT,2024-03-10 09:58:27,2023-09-19 16:35:52,2024-02-18 00:21:47,144177.250000,73728.666667,30816.666667,NaN
4,11,NaT,2023-12-09 13:13:59,NaT,NaT,2023-09-19 16:37:54,2024-01-31 10:29:55,116436.083333,76155.933333,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
28425,28464,NaT,2024-08-31 13:12:56,2024-08-31 13:21:14,2024-08-31 13:57:35,2024-08-31 12:48:13,2024-08-31 13:38:22,24.716667,25.433333,19.216667,NaN
28426,28465,NaT,2024-08-31 13:41:25,NaT,2024-08-31 15:22:35,2024-08-31 13:31:15,2024-08-31 14:33:49,10.166667,52.400000,48.766667,NaN
28427,28466,NaT,2024-08-31 14:11:00,2024-08-31 14:55:51,NaT,2024-08-31 14:03:45,2024-08-31 15:04:26,7.250000,53.433333,NaN,NaN
28428,28467,NaT,2024-08-31 14:29:10,2024-08-31 14:55:19,NaT,2024-08-31 14:12:17,NaT,16.883333,NaN,NaN,NaN


In [7]:
fact = trans_servicio.pivot_table(
    index='id',
    columns='estado',
    values='fecha_hora_key',
    aggfunc='first'
).reset_index()


fact = fact.merge(
    tiempos[
        [
            'id',
            'minutos_asignacion',
            'minutos_recogida',
            'minutos_entrega',
            'minutos_cierre'
        ]
    ],
    on='id',
    how='left'
)

fact.columns

fact = fact.rename(columns={
    'Iniciado': 'fk_fecha_solicitado',
    'Con mensajero Asignado': 'fk_fecha_asignado',
    'Recogido en origen': 'fk_fecha_recogido',
    'Cerrado': 'fk_fecha_entregado'
})

fact.drop(columns=['Con novedad'],inplace=True)

fact.head()

estado,id,fk_fecha_entregado,fk_fecha_asignado,Entregado en destino,fk_fecha_solicitado,fk_fecha_recogido,minutos_asignacion,minutos_recogida,minutos_entrega,minutos_cierre
0,7,100.0,83.0,101.0,1.0,127011.0,34649.033333,NaN,NaN,-291.916667
1,8,NaN,207.0,33623.0,2.0,5238.0,132704.633333,80359.583333,79234.283333,NaN
2,9,NaN,239.0,NaN,2.0,NaN,144182.933333,NaN,NaN,NaN
3,10,NaN,241.0,17280.0,3.0,6715.0,144177.250000,73728.666667,30816.666667,NaN
4,11,NaN,178.0,NaN,4.0,1468.0,116436.083333,76155.933333,NaN,NaN


In [8]:
fact.isnull().sum()

estado
id                          0
fk_fecha_entregado      20140
fk_fecha_asignado         728
Entregado en destino     1478
fk_fecha_solicitado         1
fk_fecha_recogido        1414
minutos_asignacion        729
minutos_recogida         1431
minutos_entrega          1497
minutos_cierre          20148
dtype: int64